In [ ]:
# Silver notebook: clean & transform Bronze into Silver
from pyspark.sql.functions import col, log1p, mean as _mean, stddev as _stddev, current_timestamp
from pyspark.sql import functions as F

# 1. Read Bronze from Delta path
bronze_path = "dbfs:/Volumes/workspace/default/project_4/bronze/creditcard_raw"
df = spark.read.format("delta").load(bronze_path)

# 2. Cast Class, drop duplicates
df = df.withColumn("Class", col("Class").cast("int")).dropDuplicates()

# 3. Stats for normalization
stats_amount = df.select(_mean("Amount").alias("meanA"), _stddev("Amount").alias("stdA")).first()
meanA, stdA = stats_amount["meanA"], stats_amount["stdA"] or 1.0

stats_time = df.select(_mean("Time").alias("meanT"), _stddev("Time").alias("stdT")).first()
meanT, stdT = stats_time["meanT"], stats_time["stdT"] or 1.0

# 4. Feature engineering
df = (
    df.withColumn("log_amount", log1p(col("Amount")))
      .withColumn("amount_zscore", (col("Amount") - F.lit(meanA)) / F.lit(stdA))
      .withColumn("time_zscore", (col("Time") - F.lit(meanT)) / F.lit(stdT))
      .withColumn("silver_ingest_ts", current_timestamp())
)

# 5. Write to Silver table
spark.sql("CREATE DATABASE IF NOT EXISTS fraud_proj")
df.write.format("delta").mode("overwrite").saveAsTable("fraud_proj.silver_creditcard")

# 6. Preview
display(spark.table("fraud_proj.silver_creditcard").limit(5))